In [1]:
import pandas as pd
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
dataset = fetch_20newsgroups(
    subset='train',
    categories=['sci.space','rec.sport.baseball'],
    remove=('headers','footers','quotes')
)

df = pd.DataFrame({
    "text": dataset.data[:200],   
    "label": dataset.target[:200]
})
df.head()

,text,label
0,"I've been saying this for quite some time, but...",0
1,Sorry for asking a question that's not entirel...,1
2,Giant's have a five man rotation of John Burk...,0
3,\n\nLot's of these small miners are no longer...,1
4,"The term ""stopper"" is generally used to refer ...",0


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df["clean_text"] = df["text"].apply(clean_text)

In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

df["processed_text"] = df["clean_text"].apply(preprocess)

In [5]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])


In [6]:
vectorizer = TfidfVectorizer(max_features=1000)

X = vectorizer.fit_transform(df["processed_text"])

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [7]:
df.to_csv("processed_text_dataset.csv", index=False)

tfidf_df.to_csv("tfidf_features.csv", index=False)